<a href="https://colab.research.google.com/github/Tamaki-Baba/text-mining/blob/main/8_1_TF_IDF_2422056%E9%A6%AC%E5%A0%B4%E7%92%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Assume that
Document:   "car insurance auto insurance"
Query:   "best car insurance"
Number of docs (N) = 1,000,000
SMART notation for tf-idf weighting variants: "ltc.ltn"

Let's fill in the value in the attached table and compute the Document-Query Similarity Score.

https://docs.google.com/document/d/15TUnKTAN1BjRP82Z1o6IUQ_y-RuRrLnGZdAwr4xvsgw/edit?usp=sharing

In [ ]:
import math
import pandas as pd

# 総文書数
N = 1_000_000

# データ
terms = ["auto", "best", "car", "insurance"]

doc_tf = {
    "auto": 1,
    "best": 0,
    "car": 1,
    "insurance": 2
}

query_tf = {
    "auto": 0,
    "best": 1,
    "car": 1,
    "insurance": 1
}

df_values = {
    "auto": 5000,
    "best": 50000,
    "car": 10000,
    "insurance": 1000
}

# tf-weight
def tf_weight(tf):
    if tf == 0:
        return 0
    return 1 + math.log10(tf)

# idf
def idf(df):
    return math.log10(N / df)

# 文書側計算
doc_weights = {}

for term in terms:
    tf_wt = tf_weight(doc_tf[term])
    idf_val = idf(df_values[term])
    wt = tf_wt * idf_val
    doc_weights[term] = wt

# cosine normalization
norm = math.sqrt(sum(w**2 for w in doc_weights.values()))

doc_norm = {
    term: doc_weights[term] / norm if norm != 0 else 0
    for term in terms
}

# クエリ側計算
query_weights = {}

for term in terms:
    tf_wt = tf_weight(query_tf[term])
    idf_val = idf(df_values[term])
    wt = tf_wt * idf_val
    query_weights[term] = wt

# Product と similarity
products = {}
similarity = 0

for term in terms:
    prod = doc_norm[term] * query_weights[term]
    products[term] = prod
    similarity += prod

# 表作成
rows = []

for term in terms:
    rows.append([
        term,
        doc_tf[term],
        round(tf_weight(doc_tf[term]), 3),
        df_values[term],
        round(idf(df_values[term]), 3),
        round(doc_weights[term], 3),
        round(doc_norm[term], 3),

        query_tf[term],
        round(tf_weight(query_tf[term]), 3),
        df_values[term],
        round(idf(df_values[term]), 3),
        round(query_weights[term], 3),
        round(query_weights[term], 3),

        round(products[term], 3)
    ])

columns = [
    "Term",
    "tf-raw(D)", "tf-wt(D)", "df", "idf",
    "wt=tf.idf", "wt-n",

    "tf-raw(Q)", "tf-wt(Q)", "df", "idf",
    "wt=tf.idf", "wt-n",

    "Product"
]

df = pd.DataFrame(rows, columns=columns)

print(df)
print("\nSimilarity Score =", round(similarity, 3))

        Term  tf-raw(D)  tf-wt(D)     df    idf  wt=tf.idf   wt-n  tf-raw(Q)  \
0       auto          1     1.000   5000  2.301      2.301  0.465          0   
1       best          0     0.000  50000  1.301      0.000  0.000          1   
2        car          1     1.000  10000  2.000      2.000  0.404          1   
3  insurance          2     1.301   1000  3.000      3.903  0.788          1   

   tf-wt(Q)     df    idf  wt=tf.idf   wt-n  Product  
0       0.0   5000  2.301      0.000  0.000    0.000  
1       1.0  50000  1.301      1.301  1.301    0.000  
2       1.0  10000  2.000      2.000  2.000    0.808  
3       1.0   1000  3.000      3.000  3.000    2.364  

Similarity Score = 3.172
